# 1. Import Libraries

In [2]:
!pip install properscoring shap

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import brier_score_loss, log_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from properscoring import crps_gaussian  # لحساب CRPS
import shap


# 2. Load Data


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import pandas as pd

# اقرأ البيانات
df = pd.read_csv("/content/drive/MyDrive/Earthquake-prediction-using-Machine-learning-models-main/Dataset/Earthquake_Data.csv")

# دمج التاريخ والوقت
df["Datetime"] = pd.to_datetime(
    df["Date(YYYY/MM/DD)"] + " " + df["Time"], errors="coerce"
)

# شيل الصفوف اللي فيها قيم ناقصة
df = df.dropna(subset=["Datetime", "Latitude", "Longitude", "Depth", "Mag"])

print(df.head())
print(df.columns.tolist())


KeyError: 'Date(YYYY/MM/DD)'

In [8]:
df = pd.read_csv("/content/drive/MyDrive/Earthquake-prediction-using-Machine-learning-models-main/Dataset/Earthquake_Data.csv")

# دمج التاريخ والوقت في خانة واحدة
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], errors="coerce")

# فلترة الأحداث الصحيحة
df = df.dropna(subset=["Datetime", "Latitude", "Longitude", "Depth", "Mag"])

print(df.head())


KeyError: 'Date'

# 3. Features & Labels

In [ ]:
X = df[["Latitude", "Longitude", "Depth"]].values
y = (df["Mag"] >= 4).astype(int)  # Binary target: زلازل >=4

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)


# 4. نموذج أساسي (Random Forest)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_prob = rf.predict_proba(X_test)[:, 1]  # احتمالية وقوع زلزال >=4


# 5. Metrics

In [ ]:
brier = brier_score_loss(y_test, y_prob)

# Log-Loss (NLL)
nll = log_loss(y_test, y_prob)

# CRPS (تقريب باستخدام Gaussian على المجاميع)
mu, sigma = np.mean(y_prob), np.std(y_prob) + 1e-6
crps_vals = [crps_gaussian(y, mu, sigma) for y in y_prob]
crps = np.mean(crps_vals)

print(f"Brier Score: {brier:.3f}")
print(f"NLL: {nll:.3f}")
print(f"CRPS: {crps:.3f}")


# 6. Calibration / Reliability Plot

In [ ]:

prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)

plt.figure(figsize=(6,6))
plt.plot(prob_pred, prob_true, marker='o', label="Model")
plt.plot([0,1], [0,1], 'k--', label="Perfect Calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration / Reliability Plot")
plt.legend()
plt.show()

# 7. Molchan Diagram

In [ ]:
thresholds = np.linspace(0,1,50)
fractions_alarm = []
fractions_detected = []

for th in thresholds:
    alarm = y_prob >= th
    fractions_alarm.append(np.mean(alarm))
    if np.sum(y_test)>0:
        fractions_detected.append(np.sum((y_test==1)&(alarm))/np.sum(y_test))
    else:
        fractions_detected.append(0)

plt.figure(figsize=(6,6))
plt.plot(fractions_alarm, fractions_detected, label="Model")
plt.plot([0,1], [0,1], 'k--', label="Random")
plt.xlabel("Fraction of alarmed space-time")
plt.ylabel("Fraction of predicted earthquakes")
plt.title("Molchan Diagram")
plt.legend()
plt.show()


# 8. SHAP Feature Importance

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values[1], X_test, feature_names=["Latitude", "Longitude", "Depth"])